## Finding Images that exists but is missing and should be in the databse

In [1]:
"""
Image-Database Cross-Check Script
--------------------------------
This script scans a directory for images, parses their filenames, and compares them
against a PostgreSQL database to identify missing entries and discrepancies.
Optimized for processing 10,000s of images efficiently.
"""

import os
import gc
import psycopg2
from psycopg2 import pool, extras
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import logging
from typing import List, Dict, Tuple, Optional, Generator
from dataclasses import dataclass, field
from enum import Enum
from dateutil import parser as date_parser
import warnings
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# ============================================================================
# CONFIGURATION SECTION - MODIFY THESE VALUES AS NEEDED
# ============================================================================

# Database Configuration
DB_CONFIG = {
    'host': '192.168.190.10',
    'port': 5432,
    'database': 'Server_LBA',
    'user': 'LBA_team',  # TO BE FILLED
    'password': 'lba',  # TO BE FILLED
    'sslmode': 'prefer',
    'min_connections': 5,  # Increased for better concurrency
    'max_connections': 20   # Increased for better throughput
}

# Directory Configuration
IMAGE_ROOT_DIRECTORY = r'W:'  # TO BE SET
TABLE_NAME = 'bildumrahmungen_aktuell'  # TO BE SET - e.g., 'images_table'

# Output Configuration
OUTPUT_EXCEL_PATH = r'G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\BIPLOV - FindingMissingDatabaseEntries'
OUTPUT_FILENAME_PREFIX = r'G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\BIPLOV - FindingMissingDatabaseEntries\missing_images_report'

# File pattern configuration
FILENAME_DELIMITER = '#'
EXPECTED_PARTS = 5  # image_number, flight_number, sortie, bestell_id, date

# Supported image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.gif'}

# Date format configuration
DATE_FORMATS = [
    '%d_%m_%Y', '%d-%m-%Y', '%Y_%m_%d', '%Y-%m-%d',
    '%d.%m.%Y', '%Y.%m.%d', '%d%m%Y', '%Y%m%d',
    '%d/%m/%Y', '%Y/%m/%d',
]

# Database date format
DB_DATE_FORMAT = '%Y-%m-%d'

# Large file handling configuration
BATCH_SIZE = 1000  # Process images in batches of 1000
MAX_WORKERS = 4    # Number of parallel threads for processing
CHUNK_SIZE = 500   # Size of chunks for dataframe processing
MEMORY_LIMIT = 0.7  # Memory usage limit (70% of available)
CACHE_ENABLED = True  # Enable caching of database queries

# Progress tracking
USE_TQDM = True  # Use tqdm for progress bars

# Logging configuration
LOG_LEVEL = logging.INFO
LOG_FORMAT = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'

# ============================================================================
# ENUMS AND DATA CLASSES
# ============================================================================

class ImageStatus(Enum):
    """Status codes for image validation results"""
    EXACT_MATCH = "exact match"
    DATE_MISMATCH = "date mismatch"
    SORTIE_MISMATCH = "sortie mismatch"
    ENTRY_MISSING = "entry missing"
    INVALID_FILENAME = "invalid filename format"
    INVALID_DATE = "invalid date format"
    DATABASE_ERROR = "database error"
    MULTIPLE_DB_ENTRIES = "multiple database entries found"

@dataclass
class ParsedFilename:
    """Data class to store parsed filename components"""
    image_number: str
    flight_number: str
    sortie: str
    bestell_id: str
    date_str: str
    full_path: str
    filename: str
    date_normalized: Optional[str] = None
    is_valid: bool = True
    error_message: str = ""

@dataclass
class DatabaseRecord:
    """Data class to store database record"""
    bildnummer: str
    sortie_ori: str
    datum: str
    found: bool = False
    datum_normalized: Optional[str] = None
    multiple_entries: bool = False

@dataclass
class ComparisonResult:
    """Data class to store comparison results"""
    parsed_file: ParsedFilename
    db_record: DatabaseRecord
    status: ImageStatus
    notes: str = ""

# ============================================================================
# MEMORY MONITORING CLASS
# ============================================================================

class MemoryMonitor:
    """Monitors memory usage and provides warnings when approaching limits"""
    
    @staticmethod
    def get_memory_usage():
        """Get current memory usage percentage"""
        try:
            import psutil
            process = psutil.Process(os.getpid())
            memory_info = process.memory_info()
            memory_percent = process.memory_percent()
            return memory_percent / 100  # Convert to fraction
        except ImportError:
            return 0  # psutil not available
    
    @staticmethod
    def check_memory_limit():
        """Check if memory usage exceeds limit"""
        usage = MemoryMonitor.get_memory_usage()
        return usage > MEMORY_LIMIT
    
    @staticmethod
    def get_memory_warning():
        """Get memory warning message if usage is high"""
        usage = MemoryMonitor.get_memory_usage()
        if usage > MEMORY_LIMIT:
            return f"High memory usage: {usage*100:.1f}%"
        return None


# ============================================================================
# DATE UTILITY CLASS
# ============================================================================

class DateNormalizer:
    """Handles date parsing and normalization"""
    
    _cache = {} if CACHE_ENABLED else None  # Cache for normalized dates
    
    @classmethod
    def normalize_date(cls, date_str: str, source: str = "filename") -> Tuple[Optional[str], Optional[str]]:
        """
        Normalize a date string to a standard format for comparison.
        Uses caching for better performance.
        Returns (normalized_date, error_message)
        """
        if not date_str or date_str == "":
            return None, "Empty date string"
        
        # Check cache first
        if cls._cache is not None:
            cache_key = f"{source}:{date_str}"
            if cache_key in cls._cache:
                return cls._cache[cache_key]
        
        # Remove any extra whitespace
        date_str = date_str.strip()
        
        # Try different parsing approaches
        normalized_date = None
        error_msg = None
        
        # Method 1: Try parsing with known formats
        for fmt in DATE_FORMATS:
            try:
                parsed_date = datetime.strptime(date_str, fmt)
                normalized_date = parsed_date.strftime(DB_DATE_FORMAT)
                if cls._cache is not None:
                    cls._cache[cache_key] = (normalized_date, None)
                return normalized_date, None
            except ValueError:
                continue
        
        # Method 2: Try using dateutil parser
        try:
            parsed_date = date_parser.parse(date_str, fuzzy=False)
            normalized_date = parsed_date.strftime(DB_DATE_FORMAT)
            if cls._cache is not None:
                cls._cache[cache_key] = (normalized_date, None)
            return normalized_date, None
        except:
            pass
        
        # Method 3: Handle special cases
        if date_str.isdigit() and len(date_str) == 8:
            # Try as YYYYMMDD
            try:
                year = int(date_str[:4])
                month = int(date_str[4:6])
                day = int(date_str[6:8])
                if 1900 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
                    parsed_date = datetime(year, month, day)
                    normalized_date = parsed_date.strftime(DB_DATE_FORMAT)
                    if cls._cache is not None:
                        cls._cache[cache_key] = (normalized_date, None)
                    return normalized_date, None
            except:
                pass
            
            # Try as DDMMYYYY
            try:
                day = int(date_str[:2])
                month = int(date_str[2:4])
                year = int(date_str[4:8])
                if 1900 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
                    parsed_date = datetime(year, month, day)
                    normalized_date = parsed_date.strftime(DB_DATE_FORMAT)
                    if cls._cache is not None:
                        cls._cache[cache_key] = (normalized_date, None)
                    return normalized_date, None
            except:
                pass
        
        error_msg = f"Could not parse date: {date_str}"
        if cls._cache is not None:
            cls._cache[cache_key] = (None, error_msg)
        return None, error_msg
    
    @classmethod
    def normalize_db_date(cls, db_date) -> Tuple[Optional[str], Optional[str]]:
        """Normalize database date to standard format."""
        if db_date is None:
            return None, "Null database date"
        
        if hasattr(db_date, 'strftime'):
            return db_date.strftime(DB_DATE_FORMAT), None
        
        if isinstance(db_date, str):
            return cls.normalize_date(db_date, source="database")
        
        return None, f"Unexpected date type: {type(db_date)}"
    
    @classmethod
    def clear_cache(cls):
        """Clear the date cache to free memory"""
        if cls._cache is not None:
            cls._cache.clear()
            gc.collect()


# ============================================================================
# CORE CLASSES
# ============================================================================

class BatchFilenameParser:
    """Handles batch parsing of image filenames for better performance"""
    
    def __init__(self):
        self.date_normalizer = DateNormalizer()
        self.logger = logging.getLogger(__name__)
    
    def parse_batch(self, file_paths: List[str]) -> List[ParsedFilename]:
        """Parse multiple filenames in batch"""
        results = []
        for file_path in file_paths:
            results.append(self.parse_filename(file_path))
        return results
    
    def parse_filename(self, file_path: str) -> ParsedFilename:
        """Parse a single image filename"""
        filename = os.path.basename(file_path)
        
        try:
            name_without_ext = os.path.splitext(filename)[0]
            parts = name_without_ext.split(FILENAME_DELIMITER)
            
            if len(parts) != EXPECTED_PARTS:
                return ParsedFilename(
                    image_number="",
                    flight_number="",
                    sortie="",
                    bestell_id="",
                    date_str="",
                    full_path=file_path,
                    filename=filename,
                    is_valid=False,
                    error_message=f"Expected {EXPECTED_PARTS} parts, got {len(parts)}"
                )
            
            date_str = parts[4].strip()
            normalized_date, date_error = self.date_normalizer.normalize_date(date_str)
            
            return ParsedFilename(
                image_number=parts[0].strip(),
                flight_number=parts[1].strip(),
                sortie=parts[2].strip(),
                bestell_id=parts[3].strip(),
                date_str=date_str,
                date_normalized=normalized_date,
                full_path=file_path,
                filename=filename,
                is_valid=(normalized_date is not None),
                error_message=date_error if date_error else ""
            )
            
        except Exception as e:
            return ParsedFilename(
                image_number="",
                flight_number="",
                sortie="",
                bestell_id="",
                date_str="",
                full_path=file_path,
                filename=filename,
                is_valid=False,
                error_message=f"Parsing error: {str(e)}"
            )


class OptimizedDatabaseChecker:
    """Optimized database checker with batch queries and connection pooling"""
    
    def __init__(self, config: dict, table_name: str):
        self.config = config
        self.table_name = table_name
        self.connection_pool = None
        self.logger = logging.getLogger(__name__)
        self.date_normalizer = DateNormalizer()
        self._create_connection_pool()
        
        # Cache for database queries
        self.query_cache = {} if CACHE_ENABLED else None
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _create_connection_pool(self):
        """Create a connection pool for database access"""
        try:
            self.connection_pool = psycopg2.pool.SimpleConnectionPool(
                self.config['min_connections'],
                self.config['max_connections'],
                host=self.config['host'],
                port=self.config['port'],
                database=self.config['database'],
                user=self.config['user'],
                password=self.config['password'],
                sslmode=self.config['sslmode']
            )
            self.logger.info(f"Database connection pool created with {self.config['min_connections']}-{self.config['max_connections']} connections")
        except Exception as e:
            self.logger.error(f"Failed to create connection pool: {e}")
            raise
    
    def get_connection(self):
        """Get a connection from the pool"""
        return self.connection_pool.getconn()
    
    def return_connection(self, conn):
        """Return a connection to the pool"""
        self.connection_pool.putconn(conn)
    
    def batch_check_images(self, parsed_files: List[ParsedFilename]) -> List[ComparisonResult]:
        """
        Check multiple images in batch for better performance
        """
        results = []
        
        # Group by image number for batch queries
        image_numbers = {}
        for i, pf in enumerate(parsed_files):
            if pf.is_valid:
                if pf.image_number not in image_numbers:
                    image_numbers[pf.image_number] = []
                image_numbers[pf.image_number].append((i, pf))
        
        # Process each image number group
        for image_number, file_indices in image_numbers.items():
            # Check cache first
            cache_key = f"image:{image_number}"
            if self.query_cache is not None and cache_key in self.query_cache:
                db_records = self.query_cache[cache_key]
                self.cache_hits += len(file_indices)
            else:
                db_records = self._get_database_records_for_image(image_number)
                if self.query_cache is not None:
                    self.query_cache[cache_key] = db_records
                self.cache_misses += len(file_indices)
            
            # Match each file with database records
            for idx, parsed_file in file_indices:
                result = self._match_with_database_records(parsed_file, db_records)
                results.append((idx, result))
        
        # Sort results by original index
        results.sort(key=lambda x: x[0])
        return [r[1] for r in results]
    
    def _get_database_records_for_image(self, image_number: str) -> List[dict]:
        """Get all database records for a specific image number"""
        conn = None
        try:
            conn = self.get_connection()
            cursor = conn.cursor(cursor_factory=extras.RealDictCursor)
            
            query = f"""
                SELECT "Bildnummer", "sortie_ori", "Datum"
                FROM {self.table_name}
                WHERE "Bildnummer" = %s
            """
            
            cursor.execute(query, (image_number,))
            results = cursor.fetchall()
            
            # Normalize dates in the results
            for record in results:
                if record['Datum']:
                    normalized, _ = self.date_normalizer.normalize_db_date(record['Datum'])
                    record['datum_normalized'] = normalized
                else:
                    record['datum_normalized'] = None
            
            return results
            
        except Exception as e:
            self.logger.error(f"Database batch query error for {image_number}: {e}")
            return []
        finally:
            if conn:
                self.return_connection(conn)
    
    def _match_with_database_records(self, parsed_file: ParsedFilename, db_records: List[dict]) -> ComparisonResult:
        """Match a parsed file with database records"""
        if not db_records:
            return ComparisonResult(
                parsed_file=parsed_file,
                db_record=DatabaseRecord(
                    bildnummer=parsed_file.image_number,
                    sortie_ori="",
                    datum="",
                    found=False
                ),
                status=ImageStatus.ENTRY_MISSING
            )
        
        multiple_entries = len(db_records) > 1
        
        # Look for exact SORTIE match first
        matching_sortie_records = [
            r for r in db_records 
            if r['sortie_ori'] and str(r['sortie_ori']) == parsed_file.sortie
        ]
        
        if not matching_sortie_records:
            # No SORTIE match found
            first_record = db_records[0]
            return ComparisonResult(
                parsed_file=parsed_file,
                db_record=DatabaseRecord(
                    bildnummer=str(first_record['Bildnummer']),
                    sortie_ori=str(first_record['sortie_ori']) if first_record['sortie_ori'] else "",
                    datum=str(first_record['Datum']) if first_record['Datum'] else "",
                    datum_normalized=first_record.get('datum_normalized'),
                    found=True,
                    multiple_entries=multiple_entries
                ),
                status=ImageStatus.SORTIE_MISMATCH,
                notes=f"DB sortie: '{first_record['sortie_ori']}', File sortie: '{parsed_file.sortie}'"
            )
        
        # Check dates among matching SORTIE records
        for record in matching_sortie_records:
            if (record.get('datum_normalized') and parsed_file.date_normalized and
                record['datum_normalized'] == parsed_file.date_normalized):
                # Exact match found
                return ComparisonResult(
                    parsed_file=parsed_file,
                    db_record=DatabaseRecord(
                        bildnummer=str(record['Bildnummer']),
                        sortie_ori=str(record['sortie_ori']),
                        datum=str(record['Datum']) if record['Datum'] else "",
                        datum_normalized=record.get('datum_normalized'),
                        found=True,
                        multiple_entries=multiple_entries
                    ),
                    status=ImageStatus.EXACT_MATCH
                )
        
        # SORTIE matches but date doesn't
        first_record = matching_sortie_records[0]
        return ComparisonResult(
            parsed_file=parsed_file,
            db_record=DatabaseRecord(
                bildnummer=str(first_record['Bildnummer']),
                sortie_ori=str(first_record['sortie_ori']),
                datum=str(first_record['Datum']) if first_record['Datum'] else "",
                datum_normalized=first_record.get('datum_normalized'),
                found=True,
                multiple_entries=multiple_entries
            ),
            status=ImageStatus.DATE_MISMATCH,
            notes=f"DB date: '{first_record['Datum']}' (norm: {first_record.get('datum_normalized')}), "
                  f"File date: '{parsed_file.date_str}' (norm: {parsed_file.date_normalized})"
        )
    
    def get_cache_stats(self):
        """Get cache hit/miss statistics"""
        if self.query_cache is None:
            return "Caching disabled"
        total = self.cache_hits + self.cache_misses
        if total == 0:
            return "No cache usage yet"
        hit_rate = (self.cache_hits / total) * 100
        return f"Cache hits: {self.cache_hits}, misses: {self.cache_misses}, hit rate: {hit_rate:.1f}%"
    
    def clear_cache(self):
        """Clear the query cache to free memory"""
        if self.query_cache is not None:
            self.query_cache.clear()
            gc.collect()
    
    def close_all_connections(self):
        """Close all database connections"""
        if self.connection_pool:
            self.connection_pool.closeall()
            self.logger.info("All database connections closed")


class LargeImageScanner:
    """Memory-efficient scanner for large directories"""
    
    def __init__(self, root_directory: str):
        self.root_directory = root_directory
        self.logger = logging.getLogger(__name__)
    
    def scan_generator(self) -> Generator[str, None, None]:
        """
        Generator that yields image paths one at a time to save memory
        """
        try:
            for root, dirs, files in os.walk(self.root_directory):
                for file in files:
                    if any(file.lower().endswith(ext) for ext in IMAGE_EXTENSIONS):
                        yield os.path.join(root, file)
        except Exception as e:
            self.logger.error(f"Error scanning directory: {e}")
            raise
    
    def get_all_images(self) -> List[str]:
        """Get all images as a list (use for smaller directories)"""
        return list(self.scan_generator())
    
    def get_image_count(self) -> int:
        """Get total number of images without loading them into memory"""
        count = 0
        for _ in self.scan_generator():
            count += 1
        return count
    
    def scan_in_batches(self, batch_size: int = BATCH_SIZE) -> Generator[List[str], None, None]:
        """
        Scan directory and yield batches of image paths
        """
        batch = []
        for image_path in self.scan_generator():
            batch.append(image_path)
            if len(batch) >= batch_size:
                yield batch
                batch = []
        
        if batch:  # Yield remaining images
            yield batch


class BatchReportGenerator:
    """Memory-efficient report generator that writes in chunks"""
    
    def __init__(self, output_path: str, filename_prefix: str):
        self.output_path = Path(output_path)
        self.filename_prefix = filename_prefix
        self.logger = logging.getLogger(__name__)
        self.temp_files = []
        
        self.output_path.mkdir(parents=True, exist_ok=True)
    
    def generate_filename(self) -> str:
        """Generate filename with timestamp"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{self.filename_prefix}_{timestamp}.xlsx"
    
    def write_batch_to_temp(self, batch_results: List[ComparisonResult], batch_num: int) -> str:
        """Write a batch of results to a temporary CSV file"""
        temp_file = self.output_path / f"temp_batch_{batch_num}_{int(time.time())}.csv"
        
        data = []
        for result in batch_results:
            file_date_display = result.parsed_file.date_str if result.parsed_file.is_valid else 'N/A'
            file_date_norm = result.parsed_file.date_normalized if result.parsed_file.date_normalized else 'N/A'
            db_date_display = result.db_record.datum if result.db_record.datum else 'N/A'
            db_date_norm = result.db_record.datum_normalized if result.db_record.datum_normalized else 'N/A'
            
            data.append({
                'Full Path': result.parsed_file.full_path,
                'Filename': result.parsed_file.filename,
                'Image Number': result.parsed_file.image_number if result.parsed_file.is_valid else 'N/A',
                'SORTIE': result.parsed_file.sortie if result.parsed_file.is_valid else 'N/A',
                'File Date (Original)': file_date_display,
                'File Date (Normalized)': file_date_norm,
                'Status': result.status.value,
                'Database Bildnummer': result.db_record.bildnummer if result.db_record.found else 'Not Found',
                'Database sortie_ori': result.db_record.sortie_ori if result.db_record.found else 'Not Found',
                'Database Datum (Original)': db_date_display,
                'Database Datum (Normalized)': db_date_norm,
                'Multiple DB Entries': 'Yes' if result.db_record.multiple_entries else 'No',
                'Notes': result.notes or result.parsed_file.error_message
            })
        
        df = pd.DataFrame(data)
        df.to_csv(temp_file, index=False)
        self.temp_files.append(temp_file)
        return str(temp_file)
    
    def merge_temp_files(self, final_file: str):
        """Merge all temporary CSV files into final Excel file"""
        if not self.temp_files:
            return
        
        self.logger.info(f"Merging {len(self.temp_files)} temporary files...")
        
        # Read and concatenate all temp files in chunks
        chunks = []
        for temp_file in self.temp_files:
            for chunk in pd.read_csv(temp_file, chunksize=CHUNK_SIZE):
                chunks.append(chunk)
                
                # If we have too many chunks, concatenate them
                if len(chunks) >= 10:
                    combined = pd.concat(chunks, ignore_index=True)
                    chunks = [combined]
        
        final_df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        
        # Write to Excel with formatting
        with pd.ExcelWriter(final_file, engine='openpyxl') as writer:
            final_df.to_excel(writer, sheet_name='Missing Images Report', index=False)
            
            # Auto-adjust column widths
            worksheet = writer.sheets['Missing Images Report']
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 70)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Add color coding
            from openpyxl.styles import PatternFill
            
            status_col = None
            for idx, col in enumerate(worksheet.columns, 1):
                if col[0].value == 'Status':
                    status_col = idx
                    break
            
            if status_col:
                for row in range(2, worksheet.max_row + 1):
                    cell = worksheet.cell(row=row, column=status_col)
                    if cell.value == 'exact match':
                        cell.fill = PatternFill(start_color='92D050', end_color='92D050', fill_type='solid')
                    elif cell.value == 'date mismatch':
                        cell.fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
                    elif cell.value == 'sortie mismatch':
                        cell.fill = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')
                    elif cell.value == 'entry missing':
                        cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
                    elif cell.value == 'invalid filename format':
                        cell.fill = PatternFill(start_color='808080', end_color='808080', fill_type='solid')
                    elif cell.value == 'invalid date format':
                        cell.fill = PatternFill(start_color='FF99CC', end_color='FF99CC', fill_type='solid')
        
        # Clean up temp files
        for temp_file in self.temp_files:
            try:
                os.remove(temp_file)
            except:
                pass
        
        self.logger.info(f"Final report generated: {final_file}")
    
    def cleanup(self):
        """Clean up any remaining temp files"""
        for temp_file in self.temp_files:
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass


class BatchImageComparator:
    """Processes images in batches for better memory management"""
    
    def __init__(self, db_checker: OptimizedDatabaseChecker, parser: BatchFilenameParser):
        self.db_checker = db_checker
        self.parser = parser
        self.logger = logging.getLogger(__name__)
    
    def process_batch(self, image_paths: List[str], batch_num: int) -> List[ComparisonResult]:
        """Process a batch of images"""
        self.logger.debug(f"Processing batch {batch_num} with {len(image_paths)} images")
        
        # Parse filenames
        parsed_files = self.parser.parse_batch(image_paths)
        
        # Separate valid and invalid files
        valid_files = [pf for pf in parsed_files if pf.is_valid]
        invalid_files = [pf for pf in parsed_files if not pf.is_valid]
        
        results = []
        
        # Process invalid files first
        for pf in invalid_files:
            if pf.error_message and "date" in pf.error_message.lower():
                status = ImageStatus.INVALID_DATE
            else:
                status = ImageStatus.INVALID_FILENAME
            
            results.append(ComparisonResult(
                parsed_file=pf,
                db_record=DatabaseRecord("", "", "", found=False),
                status=status,
                notes=pf.error_message
            ))
        
        # Process valid files in batch
        if valid_files:
            batch_results = self.db_checker.batch_check_images(valid_files)
            results.extend(batch_results)
        
        return results
    
    def process_all_batches(self, batch_generator, total_batches: int) -> Generator[List[ComparisonResult], None, None]:
        """Process all batches and yield results"""
        for batch_num, batch_paths in enumerate(batch_generator, 1):
            self.logger.info(f"Processing batch {batch_num}/{total_batches}")
            
            # Check memory before processing batch
            memory_warning = MemoryMonitor.get_memory_warning()
            if memory_warning:
                self.logger.warning(memory_warning)
                # Force garbage collection
                gc.collect()
            
            # Process batch
            batch_results = self.process_batch(batch_paths, batch_num)
            
            yield batch_results
            
            # Clear caches periodically
            if batch_num % 10 == 0:
                self.db_checker.clear_cache()
                DateNormalizer.clear_cache()
                gc.collect()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def setup_logging():
    """Setup logging configuration"""
    logging.basicConfig(
        level=LOG_LEVEL,
        format=LOG_FORMAT,
        handlers=[
            logging.FileHandler('image_crosscheck.log'),
            logging.StreamHandler()
        ]
    )

def main():
    """Main execution function with large file handling"""
    start_time = time.time()
    setup_logging()
    logger = logging.getLogger(__name__)
    
    logger.info("="*70)
    logger.info("Starting image-database cross-check (Large File Optimized)")
    logger.info("="*70)
    
    # Validate configuration
    if not DB_CONFIG['user'] or not DB_CONFIG['password']:
        logger.error("Database credentials not configured. Please set user and password in DB_CONFIG.")
        return
    
    if IMAGE_ROOT_DIRECTORY == r'path/to/images':
        logger.error("IMAGE_ROOT_DIRECTORY not configured. Please set the correct path.")
        return
    
    if TABLE_NAME == 'your_table_name':
        logger.error("TABLE_NAME not configured. Please set the correct table name.")
        return
    
    try:
        # Initialize components
        logger.info("Initializing components with large file handling...")
        logger.info(f"Batch size: {BATCH_SIZE}, Max workers: {MAX_WORKERS}, Memory limit: {MEMORY_LIMIT*100}%")
        
        parser = BatchFilenameParser()
        db_checker = OptimizedDatabaseChecker(DB_CONFIG, TABLE_NAME)
        scanner = LargeImageScanner(IMAGE_ROOT_DIRECTORY)
        comparator = BatchImageComparator(db_checker, parser)
        report_gen = BatchReportGenerator(OUTPUT_EXCEL_PATH, OUTPUT_FILENAME_PREFIX)
        
        # Count total images for progress tracking
        logger.info("Counting total images...")
        total_images = scanner.get_image_count()
        logger.info(f"Found {total_images} images in directory")
        
        if total_images == 0:
            logger.warning("No image files found in the specified directory.")
            return
        
        # Calculate batches
        total_batches = (total_images + BATCH_SIZE - 1) // BATCH_SIZE
        logger.info(f"Processing in {total_batches} batches of up to {BATCH_SIZE} images each")
        
        # Process images in batches
        logger.info("Starting batch processing...")
        batch_generator = scanner.scan_in_batches(BATCH_SIZE)
        
        all_results = []
        batch_num = 0
        
        # Use tqdm for progress bar if available
        if USE_TQDM:
            from tqdm import tqdm
            pbar = tqdm(total=total_images, desc="Processing images", unit="img")
        
        for batch_num, batch_results in enumerate(comparator.process_all_batches(batch_generator, total_batches), 1):
            # Write batch to temporary file
            report_gen.write_batch_to_temp(batch_results, batch_num)
            
            if USE_TQDM:
                pbar.update(len(batch_results))
            else:
                logger.info(f"Batch {batch_num} complete - {len(batch_results)} images processed")
            
            # Log cache stats periodically
            if batch_num % 10 == 0:
                logger.info(f"Cache stats: {db_checker.get_cache_stats()}")
        
        if USE_TQDM:
            pbar.close()
        
        # Generate final report
        logger.info("All batches processed. Generating final report...")
        final_file = report_gen.output_path / report_gen.generate_filename()
        report_gen.merge_temp_files(str(final_file))
        
        # Calculate summary statistics
        logger.info("Calculating summary statistics...")
        
        # We need to re-read some data for summary (or we could accumulate during processing)
        # For efficiency, we'll just use the first temp file to get counts
        if report_gen.temp_files:
            df_sample = pd.read_csv(report_gen.temp_files[0])
            total_valid = len(df_sample[df_sample['Status'] != 'invalid filename format'])
            
            # Simple summary
            elapsed_time = time.time() - start_time
            print("\n" + "="*70)
            print("CROSS-CHECK SUMMARY")
            print("="*70)
            print(f"Total images scanned: {total_images}")
            print(f"Batches processed: {batch_num}")
            print(f"Temporary files created: {len(report_gen.temp_files)}")
            print(f"Cache stats: {db_checker.get_cache_stats()}")
            print("-"*50)
            print(f"Processing time: {elapsed_time:.2f} seconds ({total_images/elapsed_time:.1f} images/sec)")
            print("="*70)
            print(f"Detailed report saved to: {final_file}")
        
        logger.info("Cross-check completed successfully!")
        
    except Exception as e:
        logger.error(f"Fatal error during execution: {e}", exc_info=True)
    
    finally:
        # Clean up
        if 'db_checker' in locals():
            db_checker.close_all_connections()
        if 'report_gen' in locals():
            report_gen.cleanup()
        
        # Final garbage collection
        gc.collect()
        logger.info("Cleanup complete")

if __name__ == "__main__":
    main()

2026-06-16 14:13:17,159 - __main__ - INFO - ======================================================================
2026-06-16 14:13:17,159 - __main__ - INFO - Starting image-database cross-check (Large File Optimized)
2026-06-16 14:13:17,159 - __main__ - INFO - ======================================================================
2026-06-16 14:13:17,159 - __main__ - INFO - Initializing components with large file handling...
2026-06-16 14:13:17,159 - __main__ - INFO - Batch size: 1000, Max workers: 4, Memory limit: 70.0%
2026-06-16 14:13:17,405 - __main__ - INFO - Database connection pool created with 5-20 connections
2026-06-16 14:13:17,409 - __main__ - INFO - Counting total images...
2026-06-16 14:24:28,126 - __main__ - INFO - Found 153750 images in directory
2026-06-16 14:24:28,189 - __main__ - INFO - Processing in 154 batches of up to 1000 images each
2026-06-16 14:24:28,189 - __main__ - INFO - Starting batch processing...
Processing images:   7%|▋         | 10000/153750 [02:27<25: